# Model 2_2
| Model type | Try | Comments |
|------------|-----|----------|
| 2 | 2 | GRU como capa intermedia del modelo |


In [ ]:
version = "2_2"

In [ ]:
# --- Combinar embeddings y contexto como "secuencia" ---
# Expandimos el contexto para que tenga la misma forma que los embeddings
context_seq = layers.Dense(embedding_dim, activation='relu')(context_input)
context_seq = layers.Reshape((1, embedding_dim))(context_seq)

start_seq = layers.Reshape((1, embedding_dim))(start_embed)
end_seq   = layers.Reshape((1, embedding_dim))(end_embed)

# Secuencia de tres pasos: contexto → estación origen → estación destino (teacher forcing)
sequence = layers.Concatenate(axis=1)([context_seq, start_seq, end_seq])  # shape (batch, 3, embedding_dim)

# --- Capa GRU ---
x = layers.GRU(128, return_sequences=False, dropout=0.2, recurrent_dropout=0.1)(sequence)

# --- Capa densa final ---
x = layers.Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
end_output = layers.Dense(num_stations, activation='softmax', name='end_station')(x)

model = Model(inputs=[context_input, start_input, end_input], outputs=end_output)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()
